# Citi Bike and Subway Access in New York City
Team: Pyae Pyae Thu, Elmer Rodriguez, Richard Burton

**Goal:** measure how accessible Citi Bike stations are relative to MTA subway stations, and find where both are weak.

**Data:** CityBikes API (Citi Bike stations, live snapshot), Transitland API (NYC subway stations), NYC Open Data Borough Boundaries.

**How to run:** Runtime → Run all. When asked, paste your own Transitland API key (it is never saved in this notebook).

In [ ]:
!pip install -q geopandas geopy

import requests
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from geopy.distance import geodesic
from getpass import getpass

## 1. Citi Bike stations (CityBikes API)

In [ ]:
url = "https://api.citybik.es/v2/networks/citi-bike-nyc"
data = requests.get(url).json()
df_bike = pd.DataFrame(data["network"]["stations"])
df_bike = df_bike[["id", "name", "latitude", "longitude", "free_bikes", "empty_slots", "timestamp"]]

print("Citi Bike stations (all, incl. New Jersey):", len(df_bike))
print("Snapshot time:", df_bike["timestamp"].iloc[0])
df_bike.head()

## 2. Label each station with its real borough
Uses official NYC borough boundaries. Stations outside the five boroughs (Jersey City, Hoboken) are removed, since they are served by PATH, not the subway.

In [ ]:
BOROUGH_URLS = [
    "https://data.cityofnewyork.us/api/geospatial/gthc-hcne?method=export&format=GeoJSON",
    "https://data.cityofnewyork.us/api/geospatial/tqmj-j8zm?method=export&format=GeoJSON",
]

boroughs = None
for u in BOROUGH_URLS:
    try:
        boroughs = gpd.read_file(u)
        break
    except Exception as e:
        print("Could not download from", u)

if boroughs is None:
    # Fallback: download "Borough Boundaries" GeoJSON from NYC Open Data and upload it to Colab (folder icon)
    boroughs = gpd.read_file("Borough Boundaries.geojson")

boroughs = boroughs.to_crs("EPSG:4326")

# find the borough-name column automatically
name_col = [c for c in boroughs.columns if "boro" in c.lower() and "name" in c.lower()][0]

gdf = gpd.GeoDataFrame(
    df_bike,
    geometry=gpd.points_from_xy(df_bike["longitude"], df_bike["latitude"]),
    crs="EPSG:4326",
)
gdf = gpd.sjoin(gdf, boroughs[[name_col, "geometry"]], how="inner", predicate="within")
df_bike = pd.DataFrame(gdf.drop(columns=["geometry", "index_right"])).rename(columns={name_col: "area"})

print("NYC Citi Bike stations:", len(df_bike))
print(df_bike["area"].value_counts())

## 3. Subway stations (Transitland API)
The API key is entered at run time, never stored in the code.

In [ ]:
API_KEY = getpass("Paste your Transitland API key: ")

url = "https://transit.land/api/v2/rest/stops"
headers = {"apikey": API_KEY}
params = {"feed_onestop_id": "f-dr5r-nyctsubway", "limit": 100}

all_stops = []
while True:  # the API returns results in pages, so keep going until there are none left
    data = requests.get(url, headers=headers, params=params).json()
    all_stops += data["stops"]
    after = data.get("meta", {}).get("after")
    if not after:
        break
    params["after"] = after

subway_list = []
for stop in all_stops:
    if stop.get("location_type") == 1:  # 1 = whole station (skip separate platform entries)
        subway_list.append({
            "name": stop["stop_name"],
            "lat": stop["geometry"]["coordinates"][1],
            "lon": stop["geometry"]["coordinates"][0],
        })

df_subway = pd.DataFrame(subway_list).drop_duplicates()
print("Subway stations:", len(df_subway))
df_subway.head()

## Question 1: How close is each Citi Bike station to the nearest subway station?

In [ ]:
results = []
for i, citi_row in df_bike.iterrows():
    citi_location = (citi_row["latitude"], citi_row["longitude"])
    nearest_distance = float("inf")
    nearest_stop = ""

    for j, subway_row in df_subway.iterrows():
        distance = geodesic(citi_location, (subway_row["lat"], subway_row["lon"])).kilometers
        if distance < nearest_distance:
            nearest_distance = distance
            nearest_stop = subway_row["name"]

    results.append({
        "citi_id": citi_row["id"],
        "citi_station": citi_row["name"],
        "nearest_subway": nearest_stop,
        "distance_km": round(nearest_distance, 2),
    })

df_results = pd.DataFrame(results)

# bike availability
df_bike["total_capacity"] = df_bike["free_bikes"] + df_bike["empty_slots"]
df_bike["pct_available"] = df_bike["free_bikes"] / df_bike["total_capacity"] * 100

# merge once, on station id (names can repeat)
df_combined = df_bike.merge(df_results, left_on="id", right_on="citi_id")
assert len(df_combined) == len(df_bike), "merge created duplicate rows"

print(df_results["distance_km"].describe())
df_results.sort_values("distance_km").head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df_results["distance_km"], bins=30, color="steelblue", edgecolor="black")
plt.axvline(df_results["distance_km"].mean(), color="red", linestyle="--",
            label=f"Mean: {df_results['distance_km'].mean():.2f} km")
plt.title("Distance from Citi Bike Stations to Nearest Subway Station")
plt.xlabel("Distance (km)")
plt.ylabel("Number of Citi Bike Stations")
plt.legend()
plt.show()

## Question 2: Which boroughs rely most on Citi Bike (stations more than 1 km from the subway)?

In [ ]:
distance_limit = 1.0
df_combined["high_citibike_dependency"] = df_combined["distance_km"] > distance_limit

question2_summary = (
    df_combined.groupby("area")
    .agg(
        total_citi_stations=("name", "count"),
        high_dependency_stations=("high_citibike_dependency", "sum"),
        average_distance_to_subway_km=("distance_km", "mean"),
        average_bikes_available=("free_bikes", "mean"),
        average_capacity=("total_capacity", "mean"),
    )
    .reset_index()
)
question2_summary["dependency_percent"] = (
    question2_summary["high_dependency_stations"] / question2_summary["total_citi_stations"] * 100
)
question2_summary = question2_summary.sort_values("dependency_percent", ascending=False)
question2_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(question2_summary["area"], question2_summary["dependency_percent"], color="tomato")
plt.title("Citi Bike Dependency by Borough")
plt.xlabel("Borough")
plt.ylabel("% of Citi Bike Stations More Than 1 km From Subway")
plt.tight_layout()
plt.show()

## Question 3: Which stations near the subway have low bike availability (under 20%)?

In [ ]:
df_filtered = df_combined[
    (df_combined["distance_km"] < 1.5) &
    (df_combined["pct_available"] < 20)
].sort_values("pct_available")

print("Stations near subway with under 20% bikes available:", len(df_filtered))
df_filtered[["name", "area", "distance_km", "pct_available", "free_bikes"]].head(10)

In [ ]:
df_plot = df_filtered.head(10)
plt.figure(figsize=(10, 5))
plt.bar(df_plot["name"], df_plot["pct_available"], color="skyblue")
plt.title("10 Lowest-Availability Citi Bike Stations Near the Subway")
plt.xlabel("Station")
plt.ylabel("Bike Availability (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Question 4: How many Citi Bike stations are beyond a 5-minute walk (0.4 km) of the subway?

In [ ]:
walk_limit = 0.4
far = df_results[df_results["distance_km"] > walk_limit]

print("Citi Bike stations more than 0.4 km from a subway station:", len(far))
print("Total Citi Bike stations:", len(df_results))
print("Percent:", round(len(far) / len(df_results) * 100, 1), "%")
far.sort_values("distance_km", ascending=False)[["citi_station", "nearest_subway", "distance_km"]].head(10)

In [ ]:
served = (df_results["distance_km"] <= walk_limit).map({True: "Within 0.4 km", False: "More than 0.4 km"})
served.value_counts().plot(kind="barh", color=["steelblue", "tomato"])
plt.title("Citi Bike Stations by Walking Distance to Subway")
plt.xlabel("Number of Stations")
plt.tight_layout()
plt.show()

## Question 5: Which boroughs are underserved by both (more than 1 km from subway AND under 20% bikes)?

In [ ]:
df_combined["bike_side_underserved"] = (
    (df_combined["distance_km"] > 1.0) & (df_combined["pct_available"] < 20)
)

question5_summary = (
    df_combined.groupby("area")
    .agg(
        total_citi_stations=("name", "count"),
        underserved_stations=("bike_side_underserved", "sum"),
        avg_distance_to_subway=("distance_km", "mean"),
        avg_bike_availability_pct=("pct_available", "mean"),
    )
    .reset_index()
)
question5_summary["underserved_percent"] = (
    question5_summary["underserved_stations"] / question5_summary["total_citi_stations"] * 100
)
question5_summary = question5_summary.sort_values("underserved_percent", ascending=False)
question5_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(question5_summary["area"], question5_summary["underserved_percent"], color="purple")
plt.title("Boroughs Underserved by Both Subway Access and Citi Bike Availability")
plt.xlabel("Borough")
plt.ylabel("% of Citi Bike Stations Underserved")
plt.tight_layout()
plt.show()

## Limitations
- Bike availability is a single live snapshot; it changes throughout the day.
- Distance is straight-line (geodesic), not actual walking distance.
- Only subway stations are counted; buses, PATH, and ferries are not.